# Line-by-line profiling

> **Install for this page:**
> `pip install "scope-profiler[line-profiler]"`.

scope-profiler can integrate [line_profiler](https://github.com/pyutils/line_profiler)
to give you per-source-line timing breakdowns of decorated functions,
alongside the usual region-level statistics.

## Installation

```bash
pip install "scope-profiler[line-profiler]"
```

## Enabling line profiling

Pass `use_line_profiler=True` to `setup()`:

```python
from scope_profiler import ProfileManager

ProfileManager.setup(use_line_profiler=True)
```

This selects `LineProfilerRegion` for all regions. Each region records
nanosecond timestamps and enables `line_profiler` tracing for decorated
functions and for the function containing a `with` scope.

## Example

The cells below profile into a temporary directory, so nothing is left in the
working directory. The example is first written to a script file and then run
with `%run`: `line_profiler` reports the source of a function from its file,
and the command-line viewer further down reads the source lines from that file
too, which a function defined directly in a notebook cell does not have.

In [ ]:
import os
import tempfile
from pathlib import Path

WORKDIR = Path(tempfile.mkdtemp(prefix="scope-profiler-lines-"))
os.chdir(WORKDIR)

In [ ]:
%%writefile ex_line_profiling.py
import math
import random

from scope_profiler import ProfileManager

ProfileManager.setup(use_line_profiler=True)


@ProfileManager.profile("compute")
def compute(N=50_000):
    s = 0.0
    for _ in range(N):
        x = random.random()
        s += math.sin(x) * math.sqrt(x + 1.0)
    return s


@ProfileManager.profile("allocate")
def allocate(N=100_000):
    a = [i * i for i in range(N)]
    b = []
    for i in range(N):
        b.append(i * i)
    return a, b


def run_scope_only():
    with ProfileManager.region("scope_only"):
        total = sum(range(100_000))
    return total


compute()
allocate()
run_scope_only()
ProfileManager.finalize(verbose_line_profiler=True)

In [ ]:
%run ex_line_profiling.py

The output of the run is the region summary followed by a standard `line_profiler`
table for every profiled function (`finalize(verbose_line_profiler=True)`
prints the tables; a plain `finalize()` only prints the region summary).

The line-by-line table shows, for each source line:

| Column      | Meaning                                        |
| ----------- | ---------------------------------------------- |
| **Hits**    | Number of times the line was executed          |
| **Time**    | Total time spent on that line (in timer units) |
| **Per Hit** | Average time per execution                     |
| **% Time**  | Fraction of the function's total time          |

## Decorator vs. context manager

- **Decorator** (`@ProfileManager.profile`) — automatically registers
  the function with `line_profiler`. This is the primary use case.
- **Context manager** (`with ProfileManager.region()`) —
  enables/disables the profiler around the block and automatically registers
  the active caller function. Lines executed in the scope are included even
  when the function is not decorated. Passing `functions=[...]` remains
  useful when the scope should also profile other functions it calls.

## Accessing stats programmatically

In [ ]:
from scope_profiler import ProfileManager

region = ProfileManager.get_region("compute")

# Get the line_profiler stats object
stats = region.get_stats()

# Print formatted output
region.print_stats()

`print_stats()` prints a standard `line_profiler` table.

Line-profiler data is also persisted in the HDF5 output. After reopening a
run, access the rank-local records through `ProfilingResults.line_profile`:

In [ ]:
from scope_profiler import read_h5

results = read_h5("profiling_data.h5")
for record in results.line_profile.get(0, []):
    seconds_per_unit = record["unit"]
    for line, hits, elapsed in zip(
        record["line_numbers"], record["hits"], record["times"]
    ):
        print(record["function"], line, hits, elapsed * seconds_per_unit)

The lines above are the persisted records of the example run: function, line
number, hits and seconds spent on that line.

Each record is associated with its region, source file, and function. The
stored `times` values use the `unit` reported by `line_profiler`.

The same data can be printed from the command line:

In [ ]:
!scope-profiler line-profile profiling_data.h5
!scope-profiler line-profile profiling_data.h5 --rank 0 --function compute

The second, filtered command prints only the line table of `compute`.

## Fortran and C code

Line-level profiling covers Python source only, because it relies on
`line_profiler`. Native [Fortran](/scope-profiler/guide/fortran/) and [C/C++](/scope-profiler/guide/c/) regions
record region timings and an optional source file and line, but no per-line
breakdown inside the region.

To get line-level detail for native code, narrow the region to the code you
care about (a region per loop or statement block), or profile the compiled
program with a native tool such as `perf annotate`, `gprof -l`, or
`valgrind --tool=callgrind` (build with `-g`). Use the scope-profiler regions to
find which region is hot, then use the native tool on that code.

## Overhead considerations

Line profiling adds ~50 µs per call because `line_profiler` instruments
every source line in the function. It is designed for **targeted
debugging**, not for always-on use in hot loops. See [Profiling overhead](/scope-profiler/guide/overhead/)
for benchmark data.